In [ ]:
import numpy as np 
import pandas as pd
from random import shuffle
from datasets import load_dataset, concatenate_datasets,Dataset
import os,re
from os.path import join
from pydantic import BaseModel,Field
from typing import List,Optional,Literal
import json,random
import torch

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
!pip install -U bitsandbytes>=0.43.0

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF_TOKEN")
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

In [ ]:
!huggingface-cli login --token {hf_token}

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `chatbot` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `chatbot`


In [ ]:
import wandb
wandb.login(key=wandb_key) 

wandb: Currently logged in as: mohammedehab271 (mohammedehab271-mansoura-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# !git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
# !cd LLaMA-Factory && pip install -e .

### Load Datasets

In [26]:
ds1 = load_dataset("MustafaIbrahim/medical-arabic-qa", split="train")
ds1 = ds1.shuffle(seed=42)
ds1 = ds1.select(range(2200))
ds2 = load_dataset("ashhadulislam/arabic_medical_test",split="train")
ds2 = ds2.shuffle(seed=42)
ds2 = ds2.select(range(2200))
ds3 = load_dataset("fzkuji/HealthCareMagic-100k",split="train")
ds3 = ds3.shuffle(seed=42)
ds3 = ds3.select(range(2200))

## prepare Datasets

In [27]:
def preprocess_example_1(example):
    question = example.get("question")
    answer = example.get("answer")
    
    if not question or not answer:
        return None

    question =question.strip()
    answer = answer.strip()
    answer = re.split(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}', answer)[0].strip()
    answer = re.sub(r'[\n,]+', ' ', answer)
    answer = re.sub(r'\s+', ' ', answer).strip()
    answer = re.sub(r'\s+\d+$', '', answer)
    meaningful_lines = []
    
    for line in re.split(r'\. ', answer):
        if re.search(r'[\u0600-\u06FF]', line) or re.search(r'\b[a-zA-Z]+\b', line):
            meaningful_lines.append(line.strip())
    answer = '. '.join(meaningful_lines).strip()

    return {
        "instruction": question,
        "input": "",
        "output": answer
    }

cleaned_data_1 = []
for ex in ds1:
    cleaned = preprocess_example_1(ex)
    if cleaned and cleaned["output"]:
        cleaned_data_1.append(cleaned)


In [28]:
def preprocess_example_2(example):
    text = example.get("text")
    if not text or '### Human:' not in text or '### Assistant:' not in text:
        return None
    
    try:
        question = text.split('### Human:')[1].split('### Assistant:')[0].strip()
        answer = text.split('### Assistant:')[1].strip()
    except:
        return None
    
    question =question.strip()
    answer = re.split(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}', answer)[0].strip() 
    answer = re.sub(r'[\n,]+', ' ', answer)  
    answer = re.sub(r'\s+', ' ', answer).strip()
    answer = re.sub(r'\s+\d+$', '', answer)
    
    meaningful_lines = []
    for line in re.split(r'\. ', answer):
        if re.search(r'[\u0600-\u06FF]', line) or re.search(r'\b[a-zA-Z]+\b', line):
            meaningful_lines.append(line.strip())
    answer = '. '.join(meaningful_lines).strip()
    
    if not answer:
        return None
    
    return {
        "instruction": question,
        "input": "",
        "output": answer
    }

cleaned_data_2 = []
for ex in ds2: 
    cleaned = preprocess_example_2(ex)
    if cleaned:
        cleaned_data_2.append(cleaned)

In [29]:
def preprocess_example_3(example):
    instr = example.get("instruction", "")
    user_input = example.get("input", "")
    output = example.get("output", "")

    instr = instr.strip()
    user_input = user_input.strip()
    output = output.strip()

    if not instr and not user_input:
        return None  

    forbidden_text = "If you are a doctor, please answer the medical questions based on the patient's description."
    instr = re.sub(re.escape(forbidden_text), '', instr, flags=re.IGNORECASE).strip()

    merged_instruction = f"{instr} {user_input}".strip()

    if not merged_instruction:
        return None

    output = re.sub(r'\s+', ' ', output)

    return {
        "instruction": merged_instruction,
        "input": "",
        "output": output
    }
    
cleaned_data_3 = []
for ex in ds3:
    cleaned = preprocess_example_3(ex)
    if cleaned:
        cleaned_data_3.append(cleaned)

In [ ]:
cleaned_data_1 = Dataset.from_list(cleaned_data_1)
cleaned_data_2 = Dataset.from_list(cleaned_data_2)
cleaned_data_3 = Dataset.from_list(cleaned_data_3)

final_dataset = concatenate_datasets([cleaned_data_1, cleaned_data_2, cleaned_data_3])
final_dataset = final_dataset.shuffle(seed=42)

print(final_dataset)

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 6595
})


In [31]:
len(final_dataset)

6595

## Format Finetuning Datasets

In [32]:
llm_finetunning_data = []

class MedicalAnswer(BaseModel):
    answer: str = Field(...,
        min_length=10,
        max_length=512,
        description="Clear medical answer in same language without introduction or extra commentary."
    )

system_message = "\n".join([
    "You are a professional and responsible medical assistant.",
    "Always respond must in the same language the user asked (Arabic or English) clearly and safely.",
    "If the information is incomplete, ask the user for clarification.",
    "For multi-turn conversations, remember previous context included in the chat.",
    "Advise medical consultation when necessary.",
    "Provide educational guidance only.",
    "Do not provide unsafe prescriptions."

])

for rec in final_dataset:
    merged_instruction = f"{rec['instruction']} {rec.get('input','')}".strip()
    try:
        answer = MedicalAnswer(answer=rec['output'].strip()).answer
    except:
        continue

    llm_finetunning_data.append({
        "system": system_message,
        "instruction": "### Medical Question:\n" + merged_instruction,
        "input": "",
        "output": answer,
        "history": []
    })

random.Random(101).shuffle(llm_finetunning_data)

In [33]:
len(llm_finetunning_data)

4920

In [ ]:
train_size=4000
train_ds=llm_finetunning_data[:train_size]
eval_ds=llm_finetunning_data[train_size:]

os.makedirs(join("datasets", "llamafactory-finetune-data"), exist_ok=True)

with open(join("datasets", "llamafactory-finetune-data", "train.json"), "w") as dest:
    json.dump(train_ds, dest, ensure_ascii=False, default=str)

with open(join("datasets", "llamafactory-finetune-data", "val.json"), "w", encoding="utf8") as dest:
    json.dump(eval_ds, dest, ensure_ascii=False, default=str)

## Finetuning

In [35]:
dataset_info_path = "/kaggle/working/LLaMA-Factory/data/dataset_info.json"

with open(dataset_info_path, "r", encoding="utf-8") as f:
    dataset_info = json.load(f)

In [36]:
dataset_info["news_finetune_train"] = {
    "file_name": "/kaggle/working/datasets/llamafactory-finetune-data/train.json",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output",
        "system": "system",
        "history": "history"
    }
}

dataset_info["news_finetune_val"] = {
    "file_name": "/kaggle/working/datasets/llamafactory-finetune-data/val.json",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output",
        "system": "system",
        "history": "history"
    }
}

In [37]:
with open(dataset_info_path, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=4, ensure_ascii=False)

In [38]:
%%writefile /kaggle/working/LLaMA-Factory/examples/train_qlora/news_finetune.yaml

### model
model_name_or_path: Qwen/Qwen3-4B-Instruct-2507
quantization_bit: 4
quantization_method: bnb
double_quantization: true
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 8
lora_target: all

### dataset
dataset: news_finetune_train
eval_dataset: news_finetune_val
template: qwen3_nothink
cutoff_len: 1024
# max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 16

### output
output_dir: /kaggle/working/finetuned_model
logging_steps: 10
save_steps: 500
plot_loss: true
# overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 2.0e-5
num_train_epochs: 2.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
# val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 100

report_to: wandb
run_name: medical-finetune-llamafactory

push_to_hub: true
export_hub_model_id: "mohammedehab100/medical-chatbot"
hub_private_repo: true
hub_strategy: checkpoint


Overwriting /kaggle/working/LLaMA-Factory/examples/train_qlora/news_finetune.yaml


In [39]:
!cd LLaMA-Factory/ && llamafactory-cli train /kaggle/working/LLaMA-Factory/examples/train_qlora/news_finetune.yaml

2026-03-03 17:42:02.708595: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772559722.731317     503 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772559722.737855     503 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772559722.755023     503 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772559722.755049     503 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772559722.755054     503 computation_placer.cc:177] computation placer alr